In [1]:
import csv
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
import os
import glob

RANDOM_SEED = 42

# Dataset Path

In [2]:
model_save_path = 'model/keypoint_classifier/keypoint_sequence_classifier.keras'
tflite_save_path = 'model/keypoint_classifier/keypoint_sequence_classifier.tflite'

# Parameters

In [3]:
SEQUENCE_LENGTH = 25
FEATURES_PER_FRAME = 80  # 42 hand + 10 face + 8 pose + 20 relative

# Load Dataset

In [4]:
# Load Dataset
csv_files = sorted(glob.glob('Words-Dataset/*_sequence.csv'))
X_sequences = []
y_sequences = []
for csv_file in csv_files:
    data = np.loadtxt(csv_file, delimiter=',', dtype='float32')
    X_sequences.append(data[:, 1:].reshape(-1, SEQUENCE_LENGTH, FEATURES_PER_FRAME))
    y_sequences.extend(data[:, 0])

X_dataset = np.concatenate(X_sequences, axis=0)
y_dataset = np.array(y_sequences, dtype=np.int64)
y_dataset -= 1  # Convert from 1-based to 0-based indexing for TensorFlow

# Train/val/test split (75/12.5/12.5) with stratify when possible
class_counts = np.bincount(y_dataset)
min_class_count = class_counts.min() if class_counts.size > 0 else 0
if min_class_count < 2:
    print("Warning: At least one class has <2 samples. Disabling stratify.")
    stratify_main = None
    stratify_temp = None
else:
    stratify_main = y_dataset
    stratify_temp = None  # Will set after temp split

X_train, X_temp, y_train, y_temp = train_test_split(
    X_dataset, y_dataset, test_size=0.25, random_state=RANDOM_SEED, stratify=stratify_main
 )
if stratify_main is not None:
    stratify_temp = y_temp
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=RANDOM_SEED, stratify=stratify_temp
 )

# Load labels to determine number of classes
with open('Word-Label/keypoint_sequence_classifier_label.csv', encoding='utf-8-sig') as f:
    keypoint_sequence_classifier_labels = csv.reader(f)
    keypoint_sequence_classifier_labels = [row[0] for row in keypoint_sequence_classifier_labels]
NUM_CLASSES = len(keypoint_sequence_classifier_labels)
print(f"Number of classes: {NUM_CLASSES}")
print(f"Labels: {keypoint_sequence_classifier_labels}")

Number of classes: 155
Labels: ['你好', '學校', '同學', '屋企人', '鐘意', '唔鐘意', '點解', '彩虹', '謝謝', '等等', '對不起', '聾人', '我', '健聽', '\u2060手語', '現在', '高級', '認識/見面', '開心', '再見', '香港', '早餐', '星期一', '護士', 'ok', '什麼', '麵', '紙巾', '有', '類別', '願望', '星期五', '好', '懲罰', '上個星期', '人', '是', '不是', '需要', '不需要', '幫忙', '醫生', '咖啡', '想', '不想', '爸爸', '媽媽', '父母', '哥哥', '弟弟', '姐姐', '妹妹', '星期日', '爺爺', '星期二', '最後', '分鐘', '兒子', '女兒', '老公', '老婆', '頭痛', '頭盔', '我們', '鋼琴', '你', '喝', '句子', '哪裡', '溫暖', '詞語', '工作', '標籤', '摩托車', '出糧', '不開心', '厲害', '劍擊', '同事', '文職', '網絡', '運動', '學習', '緊張', '茶', '腹瀉', '義工', '功課', '零', '一', '二', '三', '四', '睡覺', '六', '七', '牛奶', '九', '助聽器', '盲', '水', '肚餓', '渴', '飽', '美味', '朋友', '愛', '介紹', '傷心', '生氣', '害怕', '累', '病', '痛', '舒服', '冷', '熱', '大', '小', '遠', '近', '快', '慢', '新', '舊', '平', '貴', '美麗', '聰明', '強壯', '弱', '左', '右', '上面', '下', '前', '後', '最鍾意', '嘗試', '買', '教', '睇', '電話', '電影', '吃', '老師', '會去', '醫院', '假期', '責任', '一齊', '感覺', '帶', '失敗', '成功']


# Build GRU Model

In [5]:
if 'NUM_CLASSES' not in locals() or NUM_CLASSES == 0:
    print("No classes found. Please collect data first.")
else:
    from tensorflow.keras import layers
    model = tf.keras.models.Sequential([
        layers.Bidirectional(layers.GRU(256, return_sequences=True), input_shape=(SEQUENCE_LENGTH, FEATURES_PER_FRAME)),
        layers.Dropout(0.3),
        
        layers.Bidirectional(layers.GRU(128)),
        layers.Dropout(0.3),
        
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(NUM_CLASSES, activation='softmax')
    ])

    model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 bidirectional (Bidirection  (None, 25, 512)           519168    
 al)                                                             
                                                                 
 dropout (Dropout)           (None, 25, 512)           0         
                                                                 
 bidirectional_1 (Bidirecti  (None, 256)               493056    
 onal)                                                           
                                                                 
 dropout_1 (Dropout)         (None, 256)               0         
                                                                 
 dense (Dense)               (None, 128)               32896     
                                                                 
 batch_normalization (Batch  (None, 128)               5

# Compile and Train Model

In [6]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

cp_callback = tf.keras.callbacks.ModelCheckpoint(model_save_path, verbose=1, save_weights_only=False)
es_callback = tf.keras.callbacks.EarlyStopping(patience=20, verbose=1, restore_best_weights=True)

model.fit(
    X_train, y_train,
    epochs=1000,
    batch_size=32,
    validation_data=(X_val, y_val),
    callbacks=[cp_callback, es_callback],
)

Epoch 1/1000
5375/5376 [============================>.] - ETA: 0s - loss: 0.3816 - accuracy: 0.9140
Epoch 1: saving model to model/keypoint_classifier/keypoint_sequence_classifier.keras
5376/5376 [==============================] - 337s 62ms/step - loss: 0.3816 - accuracy: 0.9140 - val_loss: 0.1942 - val_accuracy: 0.9518
Epoch 2/1000
5376/5376 [==============================] - ETA: 0s - loss: 0.0440 - accuracy: 0.9876
Epoch 2: saving model to model/keypoint_classifier/keypoint_sequence_classifier.keras
5376/5376 [==============================] - 295s 55ms/step - loss: 0.0440 - accuracy: 0.9876 - val_loss: 0.0412 - val_accuracy: 0.9861
Epoch 3/1000
5376/5376 [==============================] - ETA: 0s - loss: 0.0265 - accuracy: 0.9926
Epoch 3: saving model to model/keypoint_classifier/keypoint_sequence_classifier.keras
5376/5376 [==============================] - 319s 59ms/step - loss: 0.0265 - accuracy: 0.9926 - val_loss: 0.0065 - val_accuracy: 0.9979
Epoch 4/1000
5375/5376 [==========

KeyboardInterrupt: 

# Evaluate Model

In [ ]:
val_loss, val_acc = model.evaluate(X_test, y_test)
print(f'Validation Loss: {val_loss}, Validation Accuracy: {val_acc}')

1156/1156 [==============================] - 24s 20ms/step - loss: 0.0052 - accuracy: 0.9986
Validation Loss: 0.0051566907204687595, Validation Accuracy: 0.9986207485198975


# Convert to TFLite

In [ ]:
model.save(model_save_path)

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS, tf.lite.OpsSet.SELECT_TF_OPS]
converter._experimental_lower_tensor_list_ops = False
tflite_model = converter.convert()

with open(tflite_save_path, 'wb') as f:
    f.write(tflite_model)

print("TFLite model saved.")

INFO:tensorflow:Assets written to: /var/folders/wt/bj47w0pj3h529wfvv7wn357w0000gn/T/tmpwgdntbfg/assets


INFO:tensorflow:Assets written to: /var/folders/wt/bj47w0pj3h529wfvv7wn357w0000gn/T/tmpwgdntbfg/assets


TFLite model saved.


2026-01-04 23:55:14.693566: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
2026-01-04 23:55:14.693764: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
2026-01-04 23:55:14.695442: I tensorflow/cc/saved_model/reader.cc:45] Reading SavedModel from: /var/folders/wt/bj47w0pj3h529wfvv7wn357w0000gn/T/tmpwgdntbfg
2026-01-04 23:55:14.716933: I tensorflow/cc/saved_model/reader.cc:91] Reading meta graph with tags { serve }
2026-01-04 23:55:14.716943: I tensorflow/cc/saved_model/reader.cc:132] Reading SavedModel debug info (if present) from: /var/folders/wt/bj47w0pj3h529wfvv7wn357w0000gn/T/tmpwgdntbfg
2026-01-04 23:55:14.779778: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:375] MLIR V1 optimization pass is not enabled
2026-01-04 23:55:14.801266: I tensorflow/cc/saved_model/loader.cc:231] Restoring SavedModel bundle.
2026-01-04 23:55:14.976991: I tensorflow/cc/saved_model/loader.

# Test Inference

In [ ]:
# Test Inference
interpreter = tf.lite.Interpreter(model_path=tflite_save_path)

# Add Flex delegate for TensorFlow ops
from tensorflow.lite.python.interpreter import load_delegate
try:
    delegate = load_delegate('libtensorflowlite_flex.so')
    interpreter = tf.lite.Interpreter(model_path=tflite_save_path, experimental_delegates=[delegate])
except:
    print("Flex delegate not available, using standard interpreter")

interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

interpreter.set_tensor(input_details[0]['index'], np.array([X_test[0]], dtype=np.float32))
interpreter.invoke()
result = interpreter.get_tensor(output_details[0]['index'])

print("Predicted:", np.argmax(result))
print("Actual:", y_test[0])

INFO: Created TensorFlow Lite delegate for select TF ops.
INFO: TfLiteFlexDelegate delegate: 6 nodes delegated out of 34 nodes with 3 partitions.

Exception ignored in: <function Delegate.__del__ at 0x177552440>
Traceback (most recent call last):
  File "/Users/ronald8931/Desktop/doneeee/.venv/lib/python3.10/site-packages/tensorflow/lite/python/interpreter.py", line 109, in __del__
    if self._library is not None:
AttributeError: 'Delegate' object has no attribute '_library'


Flex delegate not available, using standard interpreter
Predicted: 34
Actual: 34.0


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
